<a href="https://colab.research.google.com/github/parbin-12/Minor_project_Retinal_image_segmentation/blob/main/minor_project_phase_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset , DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision.transforms import transforms
from PIL import Image
from torchvision.transforms import InterpolationMode


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os

print(os.listdir('/content/drive/MyDrive/DRIVE'))
base_path = '/content/drive/MyDrive/DRIVE/training'

In [ ]:
#training path
print(os.listdir('/content/drive/MyDrive/DRIVE/training'))
image_dir = os.path.join(base_path, 'images')
mask_dir = os.path.join(base_path, '1st_manual')

In [ ]:
base_path = '/content/drive/MyDrive/DRIVE/training'
for folder in ['images' , '1st_manual' , 'mask']:
  path = os.path.join(base_path,folder)
  print(folder ,":",len(os.listdir(path)))
  print(os.listdir(path)[:5])
  print()

In [ ]:
#opening the image
image = Image.open('/content/drive/MyDrive/DRIVE/training/images/33_training.tif')

#information
print("origional image",image.size)
print("origional image",image.mode)
# resize
resized = image.resize((256, 256))
print("New size:", resized.size)
# display
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(resized)
plt.title("256 x 256")
plt.axis("off")

plt.show()

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

mask_transform = transforms.Compose([
    transforms.Resize(
        (256, 256),
        interpolation=InterpolationMode.NEAREST
    ),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float())
])

In [ ]:
#defining our custom dataset now
class CustomDataset(Dataset):
  def __init__(self,image_dir,mask_dir,image_transform = None ,mask_transform = None):
    self.image_dir = image_dir
    self.mask_dir = mask_dir
    self.image_transform = image_transform
    self.mask_transform = mask_transform
    self.images = os.listdir(image_dir)

  def __len__(self):
    return len(self.images)

  def __getitem__(self, index):

        image_name = self.images[index]
        patient_id = image_name.split('_')[0]
        mask_name = patient_id + '_manual1.gif'
        image_path = os.path.join(
            self.image_dir,
            image_name
        )
        mask_path = os.path.join(
            self.mask_dir,
            mask_name
        )
        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.image_transform:
            image = self.image_transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)
        return image, mask


In [ ]:
dataset = CustomDataset(
    image_dir=image_dir,
    mask_dir=mask_dir,
    image_transform=image_transform,
    mask_transform=mask_transform
)

In [ ]:
image,mask = dataset[0]
print("image shape",image.shape)
print("mask shape",mask.shape)

In [ ]:
#here is the dataloader now
train_data = DataLoader(
    dataset,
    batch_size = 32,
    shuffle = True
)

In [ ]:
#here is the test data
test_data = DataLoader(
    dataset,
    batch_size =32,
    shuffle=False
)

In [ ]:
images, masks = next(iter(train_data))

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(4):

    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title(f"image {i+1}")
    plt.axis("off")

    plt.subplot(2, 4, i + 5)
    plt.imshow(masks[i].squeeze(), cmap="gray")
    plt.title(f"mask {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
%pip install segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="vgg16",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

In [ ]:
#fetching the pretrained model here
import torchvision.models as models
vgg16 = models.vgg16(pretrained=True)


In [ ]:
#feature weight freezing
for param in vgg16.features.parameters():
  param.requires_grad = False

In [ ]:
#definging our own classifier
vgg16.classifier = nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Linear(512,2)
)

In [ ]:
learning_rate = 0.01
epochs = 100

In [ ]:
# loss
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, outputs, targets):
        bce_loss = self.bce(outputs, targets)

        outputs = torch.sigmoid(outputs)
        outputs = outputs.reshape(-1)
        targets = targets.reshape(-1)

        intersection = (outputs * targets).sum()

        dice_loss = 1 - (
            (2 * intersection + 1e-6) /
            (outputs.sum() + targets.sum() + 1e-6)
        )

        return bce_loss + dice_loss

In [ ]:
fdevice = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = model.to(device)

criterion = DiceBCELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

epochs = 100

In [ ]:
for epoch in range(epochs):
    model.train()

    total_epoch_loss = 0.0
    total_correct = 0
    total_pixels = 0

    for batch_features, batch_masks in train_data:
        batch_features = batch_features.to(device)
        batch_masks = batch_masks.to(device)

        optimizer.zero_grad()

        outputs = model(batch_features)
        loss = criterion(outputs, batch_masks)

        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

        predictions = torch.sigmoid(outputs) >= 0.5
        targets = batch_masks >= 0.5

        total_correct += (predictions == targets).sum().item()
        total_pixels += targets.numel()

    avg_loss = total_epoch_loss / len(train_data)
    train_accuracy = total_correct / total_pixels

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {avg_loss:.4f} "
        f"Train Accuracy: {train_accuracy * 100:.2f}%"
    )

In [ ]:
# defining the yest accuracy here
def test_accuracy(model, test_data, device):
    model.eval()

    total_correct = 0
    total_pixels = 0

    with torch.no_grad():
        for images, masks in test_data:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            predictions = torch.sigmoid(outputs) >= 0.5
            targets = masks >= 0.5

            total_correct += (predictions == targets).sum().item()
            total_pixels += targets.numel()

    accuracy = total_correct / total_pixels

    return accuracy * 100

In [ ]:
for epoch in range(epochs):
    model.train()
    total_epoch_loss = 0.0

    for batch_features, batch_masks in train_data:
        batch_features = batch_features.to(device)
        batch_masks = batch_masks.to(device)

        optimizer.zero_grad()

        outputs = model(batch_features)
        loss = criterion(outputs, batch_masks)

        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_data)

    train_acc = test_accuracy(model, train_data, device)
    test_acc = test_accuracy(model, test_data, device)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {avg_loss:.4f} "
        f"Train Accuracy: {train_acc:.2f}% "
        f"Test Accuracy: {test_acc:.2f}%"
    )